<a href="https://colab.research.google.com/github/chetools/CHE4061_Spring2026/blob/main/McCabeThiel_NRTL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [2]:
r=Props(['Isopropanol','Water'])

In [3]:
def bubbleT_NRTL(x, P):

    T0 = np.sum(x*r.Tb(P))
    def froot(T):
        return np.sum(x*r.NRTL_gamma(x,T)*r.Pvap(T)/P)-1.

    T=sp.optimize.root_scalar(froot, x0=T0, method='secant').root
    return T, x*r.NRTL_gamma(x,T)*r.Pvap(T)/P

In [4]:
P=1e5
x1s = np.linspace(0,1,31)
y1s = []
bubbleTs = []
for x1 in x1s:
    T, (y1, _) = bubbleT_NRTL(np.array([x1, 1-x1]), P)
    bubbleTs.append(T)
    y1s.append(y1)
bubbleTs = np.array(bubbleTs)
y1s = np.array(y1s)

In [6]:
x_equil_func=sp.interpolate.PchipInterpolator(y1s, x1s)

y_eq = np.linspace(0,1,51)
x_eq = x_equil_func(y_eq)

In [20]:
F = 1. #mol/s
zF = 0.4
q = 0.9  #fraction liquid in feed
rd = 0.95
xD = 0.65
R = 0.8

D = rd*F*zF/xD
B = F - D
xB = (zF*F - xD*D)/B
Vb = (D*(R+1) - (1-q)*F)/B    #boilup ratio
x_rs = (xB/Vb + xD/(R+1))/((Vb+1)/Vb - R/(R+1))
y_rs = R/(R+1)*x_rs + xD/(R+1)

In [21]:
Nmax=50
xy_steps =[[xD,xD]]
y_equil = xD
for i in range(Nmax):
    x_equil = x_equil_func(y_equil)
    xy_steps.append([x_equil,y_equil])
    if x_equil < xB:
        break
    y_rec = R*x_equil/(R+1) + xD/(R+1)
    y_strip = (Vb+1)*x_equil/Vb - xB/Vb
    y_equil = min(y_rec,y_strip)
    xy_steps.append([x_equil,y_equil])

xy_steps = np.array(xy_steps)

In [22]:
fig = make_subplots()
fig.add_scatter(x=[xD, 0], y=[xD, xD/(R+1)], mode='lines')
fig.add_scatter(x=[xB, (Vb+xB)/(Vb+1)], y=[xB, 1],mode='lines')
fig.add_scatter(x=[x_rs, zF], y=[y_rs, zF],mode='lines')
fig.add_scatter(x=xy_steps[:,0], y=xy_steps[:,1], mode='lines')
fig.add_scatter(x=x_eq, y = y_eq, mode='lines')
fig.add_scatter(x=[0,1],y=[0,1],mode='lines',line_color='grey')
fig.update_layout(width=500, height=500, showlegend=False)